In [0]:
dbutils.widgets.text("catalog_name", "allianz_coe")
dbutils.widgets.text("gold_schema_name", "gold_c360")
dbutils.widgets.text("vault_schema_name", "dvault_faker01")
dbutils.widgets.text("control_schema_name", "audit_control")
dbutils.widgets.text("watermark_table_name", "application_watermark")

catalog_name = dbutils.widgets.get("catalog_name")
gold_schema_name = dbutils.widgets.get("gold_schema_name")
vault_schema_name = dbutils.widgets.get("vault_schema_name")
control_schema_name = dbutils.widgets.get("control_schema_name")
watermark_table_name = dbutils.widgets.get("watermark_table_name")

In [0]:
FACT_LEAD_CFG = {
    "name": "fact_lead",
    "target_table": f"{catalog_name}.{gold_schema_name}.fact_lead",
    "stage_sql": f"""
WITH wm AS (
    SELECT COALESCE(MAX(watermark), TIMESTAMP('1900-01-01 00:00:00')) AS watermark
    FROM {catalog_name}.{control_schema_name}.{watermark_table_name}
    WHERE table_name = '{catalog_name}.{gold_schema_name}.fact_lead'
),

hl AS (
    SELECT lead_hash_key, lead_id
    FROM {catalog_name}.{vault_schema_name}.hub_lead
),

-- ==========================================================
-- Lead EVENTS (historized): keep ALL sat_lead rows after watermark
-- ==========================================================
lead_event AS (
    SELECT
        s.*,
        to_timestamp(s.load_date) AS event_ts
    FROM {catalog_name}.{vault_schema_name}.sat_lead s
    WHERE to_timestamp(s.load_date) > (SELECT watermark FROM wm)
),

-- ==========================================================
-- Link selections AS-OF event_ts (prevents future links being used)
-- If you don't care about as-of correctness, you can use simple rn_link=1 logic.
-- ==========================================================

-- Lead -> Person link as-of event
lead_person_asof AS (
    SELECT * EXCEPT(rn_link) FROM (
        SELECT
            le.lead_hash_key,
            le.load_date        AS lead_load_date,
            lpl.person_hash_key,
            lpl.load_date       AS link_load_date,
            ROW_NUMBER() OVER (
                PARTITION BY le.lead_hash_key, le.load_date
                ORDER BY lpl.load_date DESC
            ) AS rn_link
        FROM lead_event le
        JOIN {catalog_name}.{vault_schema_name}.link_person_lead lpl
          ON lpl.lead_hash_key = le.lead_hash_key
         AND to_timestamp(lpl.load_date) <= le.event_ts
    ) WHERE rn_link = 1
),

-- Person -> marketing preference as-of event
person_mkt_pref_asof AS (
    SELECT * EXCEPT(rn_link) FROM (
        SELECT
            le.lead_hash_key,
            le.load_date AS lead_load_date,
            lp.person_hash_key,
            lpmp.marketing_preference_hash_key,
            lpmp.load_date AS link_load_date,
            ROW_NUMBER() OVER (
                PARTITION BY le.lead_hash_key, le.load_date
                ORDER BY lpmp.load_date DESC
            ) AS rn_link
        FROM lead_event le
        JOIN lead_person_asof lp
          ON lp.lead_hash_key = le.lead_hash_key
         AND lp.lead_load_date = le.load_date
        LEFT JOIN {catalog_name}.{vault_schema_name}.link_person_marketing_preference lpmp
          ON lpmp.person_hash_key = lp.person_hash_key
         AND to_timestamp(lpmp.load_date) <= le.event_ts
    ) WHERE rn_link = 1
),

-- Person -> marketing engagement as-of event
person_mkt_eng_asof AS (
    SELECT * EXCEPT(rn_link) FROM (
        SELECT
            le.lead_hash_key,
            le.load_date AS lead_load_date,
            lp.person_hash_key,
            lpme.marketing_engagement_hash_key,
            lpme.load_date AS link_load_date,
            ROW_NUMBER() OVER (
                PARTITION BY le.lead_hash_key, le.load_date
                ORDER BY lpme.load_date DESC
            ) AS rn_link
        FROM lead_event le
        JOIN lead_person_asof lp
          ON lp.lead_hash_key = le.lead_hash_key
         AND lp.lead_load_date = le.load_date
        LEFT JOIN {catalog_name}.{vault_schema_name}.link_person_marketing_engagement lpme
          ON lpme.person_hash_key = lp.person_hash_key
         AND to_timestamp(lpme.load_date) <= le.event_ts
    ) WHERE rn_link = 1
)

SELECT
    -- SKs (default unknown)
    COALESCE(dp.person_sk, -1)     AS person_sk,
    COALESCE(dd.date_sk, -1)       AS date_sk,
    COALESCE(dm.marketing_sk, -1)  AS marketing_sk,

    -- Degenerate keys
    dp.person_id                   AS person_id,
    hl.lead_id                     AS lead_id,

    -- Lead attributes / measures
    le.interested_level            AS interested_level,
    CAST(le.person_score AS FLOAT) AS person_score,
    le.person_status               AS person_status,

    -- As per your previous mapping: you used converted_date as lead_creation_ts
    le.converted_date              AS lead_creation_ts,

    -- Fact event timestamp
    le.event_ts                    AS load_ts,

    current_timestamp()            AS created_ts,
    'ETL_SYSTEM'                   AS created_by

FROM lead_event le
JOIN hl
  ON le.lead_hash_key = hl.lead_hash_key

-- Lead -> Person
LEFT JOIN lead_person_asof lpa
  ON lpa.lead_hash_key = le.lead_hash_key
 AND lpa.lead_load_date = le.load_date

LEFT JOIN {catalog_name}.{vault_schema_name}.hub_person hp
  ON hp.person_hash_key = lpa.person_hash_key

-- dim_person SCD2 as-of join (CORRECT)
LEFT JOIN {catalog_name}.{gold_schema_name}.dim_person dp
  ON dp.person_id = hp.person_id
 AND le.event_ts BETWEEN dp.effective_from_ts AND dp.effective_to_ts

-- dim_date join
-- ✅ Choose ONE: lead event date or converted_date date
-- Option 1 (recommended for historized grain): use lead load_date
--LEFT JOIN {catalog_name}.{gold_schema_name}.dim_date dd
--  ON to_date(dd.full_date, 'yyyy/MM/dd') = to_date(le.load_date)

-- Option 2 (if you want date_sk for converted_date instead):
LEFT JOIN {catalog_name}.{gold_schema_name}.dim_date dd 
    ON to_date(dd.full_date,'yyyy/MM/dd') = to_date(le.converted_date)

-- Marketing preference + engagement hubs
LEFT JOIN person_mkt_pref_asof pmp
  ON pmp.lead_hash_key = le.lead_hash_key
 AND pmp.lead_load_date = le.load_date

LEFT JOIN {catalog_name}.{vault_schema_name}.hub_marketing_preference mp
  ON mp.marketing_preference_hash_key = pmp.marketing_preference_hash_key

LEFT JOIN person_mkt_eng_asof pme
  ON pme.lead_hash_key = le.lead_hash_key
 AND pme.lead_load_date = le.load_date

LEFT JOIN {catalog_name}.{vault_schema_name}.hub_marketing_engagement me
  ON me.marketing_engagement_hash_key = pme.marketing_engagement_hash_key

-- dim_marketing composite BK SCD2 as-of join (CORRECT)
LEFT JOIN {catalog_name}.{gold_schema_name}.dim_marketing dm
  ON dm.marketing_preference_id = mp.marketing_preference_id
 AND dm.marketing_engagement_id <=> me.marketing_engagement_id
 AND le.event_ts BETWEEN dm.effective_from_ts AND dm.effective_to_ts
""",
    "increment_keys": ['lead_id']
}

In [0]:
FACT_QUOTE_CFG = {
    "name": "fact_quote",
    "target_table": f"{catalog_name}.{gold_schema_name}.fact_quote",
    "stage_sql": f"""
    WITH wm AS (
  SELECT COALESCE(MAX(watermark), TIMESTAMP('1900-01-01 00:00:00')) AS watermark
  FROM {catalog_name}.{control_schema_name}.{watermark_table_name}
  WHERE table_name = '{catalog_name}.{gold_schema_name}.fact_quote'
),

hq AS (
  SELECT quote_hash_key, quote_id
  FROM {catalog_name}.{vault_schema_name}.hub_quote
),

-- Ranked links (avoid fan-out): keep only latest row per key in the CTE
rl_person_account AS (
  SELECT * EXCEPT(rn_link) FROM (
    SELECT
      lpa.*,
      ROW_NUMBER() OVER (PARTITION BY lpa.person_hash_key ORDER BY lpa.load_date DESC) AS rn_link
    FROM {catalog_name}.{vault_schema_name}.link_person_account lpa
  ) WHERE rn_link = 1
),

rl_quote_person AS (
  SELECT * EXCEPT(rn_link) FROM (
    SELECT
      lqp.*,
      ROW_NUMBER() OVER (PARTITION BY lqp.quote_hash_key ORDER BY lqp.load_date DESC) AS rn_link
    FROM {catalog_name}.{vault_schema_name}.link_quote_person lqp
  ) WHERE rn_link = 1
),

rl_quote_product AS (
  SELECT * EXCEPT(rn_link) FROM (
    SELECT
      lpr.*,
      ROW_NUMBER() OVER (PARTITION BY lpr.quote_hash_key ORDER BY lpr.load_date DESC) AS rn_link
    FROM {catalog_name}.{vault_schema_name}.link_quote_product lpr
  ) WHERE rn_link = 1
),

rl_product_home AS (
  SELECT * EXCEPT(rn_link) FROM (
    SELECT
      lph.*,
      ROW_NUMBER() OVER (PARTITION BY lph.product_hash_key ORDER BY lph.load_date DESC) AS rn_link
    FROM {catalog_name}.{vault_schema_name}.link_product_home lph
  ) WHERE rn_link = 1
),

rl_product_motor AS (
  SELECT * EXCEPT(rn_link) FROM (
    SELECT
      lpm.*,
      ROW_NUMBER() OVER (PARTITION BY lpm.product_hash_key ORDER BY lpm.load_date DESC) AS rn_link
    FROM {catalog_name}.{vault_schema_name}.link_product_motor lpm
  ) WHERE rn_link = 1
),

rl_customer_person AS (
  SELECT * EXCEPT(rn_link) FROM (
    SELECT
      lcp.*,
      ROW_NUMBER() OVER (PARTITION BY lcp.person_hash_key ORDER BY lcp.load_date DESC) AS rn_link
    FROM {catalog_name}.{vault_schema_name}.link_customer_person lcp
  ) WHERE rn_link = 1
),

rl_person_mkt_pref AS (
  SELECT * EXCEPT(rn_link) FROM (
    SELECT
      lpmp.*,
      ROW_NUMBER() OVER (PARTITION BY lpmp.person_hash_key ORDER BY lpmp.load_date DESC) AS rn_link
    FROM {catalog_name}.{vault_schema_name}.link_person_marketing_preference lpmp
  ) WHERE rn_link = 1
),

rl_person_mkt_eng AS (
  SELECT * EXCEPT(rn_link) FROM (
    SELECT
      lpme.*,
      ROW_NUMBER() OVER (PARTITION BY lpme.person_hash_key ORDER BY lpme.load_date DESC) AS rn_link
    FROM {catalog_name}.{vault_schema_name}.link_person_marketing_engagement lpme
  ) WHERE rn_link = 1
),

-- Latest satellites AFTER watermark; compare timestamp-to-timestamp
latest_quote AS (
  SELECT * EXCEPT(rn) FROM (
    SELECT
      s.*,
      ROW_NUMBER() OVER (PARTITION BY s.quote_hash_key ORDER BY s.load_date DESC) AS rn
    FROM {catalog_name}.{vault_schema_name}.sat_quote s
    WHERE to_timestamp(s.load_date) > (SELECT watermark FROM wm)
  ) WHERE rn = 1
),

quote_event AS (
  SELECT
    lq.*,
    to_timestamp(lq.load_date) AS event_ts
  FROM latest_quote lq
)

SELECT
  -- SKs
  COALESCE(dp.person_sk, -1)        AS person_sk,
  COALESCE(dc.customer_sk, -1)      AS customer_sk,
  COALESCE(da.account_sk, -1)       AS account_sk,
  COALESCE(dm.marketing_sk, -1)     AS marketing_sk,
  COALESCE(dd.date_sk, -1)          AS date_sk,
  COALESCE(dh.home_sk, -1)          AS home_sk,
  COALESCE(dmotor.motor_sk, -1)     AS motor_sk,

  -- Degenerate keys / identifiers
  dp.person_id                      AS person_id,
  hq.quote_id                       AS quote_id,

  -- Quote attributes / measures
  qe.quote_number                   AS quote_number,
  qe.quote_status                   AS quote_status,
  CAST(qe.gross_revenue AS DECIMAL(18,4)) AS quote_gross_revenue_amt,
  CAST(qe.net_revenue   AS DECIMAL(18,4)) AS quote_net_revenue_amt,
  CAST(qe.renewal_amt_current_period AS DECIMAL(18,4)) AS quote_renewal_current_period_amt,
  CAST(qe.renewal_amt_next_period    AS DECIMAL(18,4)) AS quote_renewal_next_period_amt,

  qe.event_ts                        AS load_ts,
  current_timestamp()                AS created_ts,
  'ETL_SYSTEM'                       AS created_by

FROM quote_event qe
JOIN hq
  ON qe.quote_hash_key = hq.quote_hash_key

-- Quote -> Person
LEFT JOIN rl_quote_person rqp
  ON rqp.quote_hash_key = hq.quote_hash_key
LEFT JOIN {catalog_name}.{vault_schema_name}.hub_person hp
  ON hp.person_hash_key = rqp.person_hash_key

-- Dim person (SCD2 as-of join)
LEFT JOIN {catalog_name}.{gold_schema_name}.dim_person dp
  ON dp.person_id = hp.person_id
 AND qe.event_ts BETWEEN dp.effective_from_ts AND dp.effective_to_ts

-- Dim date (you said full_date is string 'yyyy/MM/dd' in earlier query; keep as-is)
LEFT JOIN {catalog_name}.{gold_schema_name}.dim_date dd
  ON to_date(dd.full_date, 'yyyy/MM/dd') = to_date(qe.load_date)

-- Person -> Customer -> Dim customer
LEFT JOIN rl_customer_person rcp
  ON rcp.person_hash_key = hp.person_hash_key
LEFT JOIN {catalog_name}.{vault_schema_name}.hub_customer hc
  ON hc.customer_hash_key = rcp.customer_hash_key
LEFT JOIN {catalog_name}.{gold_schema_name}.dim_customer dc
  ON dc.customer_id = hc.customer_id
 AND qe.event_ts BETWEEN dc.effective_from_ts AND dc.effective_to_ts

-- Person -> Account -> Dim account
LEFT JOIN rl_person_account rpa
  ON rpa.person_hash_key = hp.person_hash_key
LEFT JOIN {catalog_name}.{vault_schema_name}.hub_account ha
  ON ha.account_hash_key = rpa.account_hash_key
LEFT JOIN {catalog_name}.{gold_schema_name}.dim_account da
  ON da.account_id = ha.account_id
 AND qe.event_ts BETWEEN da.effective_from_ts AND da.effective_to_ts

-- Quote -> Product
LEFT JOIN rl_quote_product rpp
  ON rpp.quote_hash_key = hq.quote_hash_key

-- Product -> Home -> Dim home
LEFT JOIN rl_product_home rph
  ON rph.product_hash_key = rpp.product_hash_key
LEFT JOIN {catalog_name}.{vault_schema_name}.hub_home hh
  ON hh.home_hash_key = rph.home_hash_key
LEFT JOIN {catalog_name}.{gold_schema_name}.dim_home dh
  ON trim(upper(dh.home_id)) = trim(upper(hh.home_id))
 AND qe.event_ts BETWEEN dh.effective_from_ts AND dh.effective_to_ts

-- Product -> Motor -> Dim motor
LEFT JOIN rl_product_motor rpm
  ON rpm.product_hash_key = rpp.product_hash_key
LEFT JOIN {catalog_name}.{vault_schema_name}.hub_motor hmotor
  ON hmotor.motor_hash_key = rpm.motor_hash_key
LEFT JOIN {catalog_name}.{gold_schema_name}.dim_motor dmotor
  ON trim(upper(dmotor.motor_id)) = trim(upper(hmotor.motor_id))  -- FIXED alias: hmotor
 AND qe.event_ts BETWEEN dmotor.effective_from_ts AND dmotor.effective_to_ts

-- Marketing preference + engagement -> Dim marketing (composite BK)
LEFT JOIN rl_person_mkt_pref rpmp
  ON rpmp.person_hash_key = hp.person_hash_key
LEFT JOIN {catalog_name}.{vault_schema_name}.hub_marketing_preference hmp
  ON hmp.marketing_preference_hash_key = rpmp.marketing_preference_hash_key

LEFT JOIN rl_person_mkt_eng rpme
  ON rpme.person_hash_key = hp.person_hash_key
LEFT JOIN {catalog_name}.{vault_schema_name}.hub_marketing_engagement hme
  ON hme.marketing_engagement_hash_key = rpme.marketing_engagement_hash_key

LEFT JOIN {catalog_name}.{gold_schema_name}.dim_marketing dm
  ON dm.marketing_preference_id = hmp.marketing_preference_id
 AND dm.marketing_engagement_id <=> hme.marketing_engagement_id
 AND qe.event_ts BETWEEN dm.effective_from_ts AND dm.effective_to_ts
""",
    "increment_keys": ['quote_id']
}

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:720)
	at com.data

In [0]:
FACT_POLICY_CFG = {
    "name": "fact_policy",
    "target_table": f"{catalog_name}.{gold_schema_name}.fact_policy",
    "stage_sql": f"""
WITH wm AS (
    SELECT COALESCE(MAX(watermark), TIMESTAMP('1900-01-01 00:00:00')) AS watermark
    FROM {catalog_name}.{control_schema_name}.{watermark_table_name}
    WHERE table_name = '{catalog_name}.{gold_schema_name}.fact_policy'
),

-- Hub Policy
hpoc AS (
    SELECT policy_hash_key, policy_id
    FROM {catalog_name}.{vault_schema_name}.hub_policy
),

-- ----------------------------
-- Ranked links (avoid fan-out): keep only latest row per key
-- ----------------------------
rl_policy_customer AS (
    SELECT * EXCEPT(rn_link) FROM (
        SELECT
            lpc.*,
            ROW_NUMBER() OVER (PARTITION BY lpc.policy_hash_key ORDER BY lpc.load_date DESC) AS rn_link
        FROM {catalog_name}.{vault_schema_name}.link_policy_customer lpc
    ) WHERE rn_link = 1
),

rl_policy_product AS (
    SELECT * EXCEPT(rn_link) FROM (
        SELECT
            lpp.*,
            ROW_NUMBER() OVER (PARTITION BY lpp.policy_hash_key ORDER BY lpp.load_date DESC) AS rn_link
        FROM {catalog_name}.{vault_schema_name}.link_policy_product lpp
    ) WHERE rn_link = 1
),

rl_product_home AS (
    SELECT * EXCEPT(rn_link) FROM (
        SELECT
            lph.*,
            ROW_NUMBER() OVER (PARTITION BY lph.product_hash_key ORDER BY lph.load_date DESC) AS rn_link
        FROM {catalog_name}.{vault_schema_name}.link_product_home lph
    ) WHERE rn_link = 1
),

rl_product_motor AS (
    SELECT * EXCEPT(rn_link) FROM (
        SELECT
            lpm.*,
            ROW_NUMBER() OVER (PARTITION BY lpm.product_hash_key ORDER BY lpm.load_date DESC) AS rn_link
        FROM {catalog_name}.{vault_schema_name}.link_product_motor lpm
    ) WHERE rn_link = 1
),

rl_customer_person AS (
    SELECT * EXCEPT(rn_link) FROM (
        SELECT
            lcp.*,
            ROW_NUMBER() OVER (PARTITION BY lcp.customer_hash_key ORDER BY lcp.load_date DESC) AS rn_link
        FROM {catalog_name}.{vault_schema_name}.link_customer_person lcp
    ) WHERE rn_link = 1
),

rl_person_account AS (
    SELECT * EXCEPT(rn_link) FROM (
        SELECT
            lpa.*,
            ROW_NUMBER() OVER (PARTITION BY lpa.person_hash_key ORDER BY lpa.load_date DESC) AS rn_link
        FROM {catalog_name}.{vault_schema_name}.link_person_account lpa
    ) WHERE rn_link = 1
),

rl_person_mkt_pref AS (
    SELECT * EXCEPT(rn_link) FROM (
        SELECT
            lpmp.*,
            ROW_NUMBER() OVER (PARTITION BY lpmp.person_hash_key ORDER BY lpmp.load_date DESC) AS rn_link
        FROM {catalog_name}.{vault_schema_name}.link_person_marketing_preference lpmp
    ) WHERE rn_link = 1
),

rl_person_mkt_eng AS (
    SELECT * EXCEPT(rn_link) FROM (
        SELECT
            lpme.*,
            ROW_NUMBER() OVER (PARTITION BY lpme.person_hash_key ORDER BY lpme.load_date DESC) AS rn_link
        FROM {catalog_name}.{vault_schema_name}.link_person_marketing_engagement lpme
    ) WHERE rn_link = 1
),

-- ----------------------------
-- Latest Policy satellite rows after watermark (timestamp-to-timestamp)
-- Keep rn=1 per policy_hash_key for "current snapshot per policy" in this increment.
-- If you want historized fact per sat version, remove rn=1 filter.
-- ----------------------------
latest_policy AS (
    SELECT * EXCEPT(rn) FROM (
        SELECT
            s.*,
            ROW_NUMBER() OVER (PARTITION BY s.policy_hash_key ORDER BY s.load_date DESC) AS rn
        FROM {catalog_name}.{vault_schema_name}.sat_policy s
        WHERE to_timestamp(s.load_date) > (SELECT watermark FROM wm)
    ) WHERE rn = 1
),

policy_event AS (
    SELECT
        lp.*,
        to_timestamp(lp.load_date) AS event_ts
    FROM latest_policy lp
)

SELECT
    -- Policy dimension SK (as-of join)
    COALESCE(dpo.policy_sk, -1)     AS policy_sk,

    -- Person (via customer -> person)
    COALESCE(dp.person_sk, -1)      AS person_sk,
    hp.person_id                    AS person_id,

    -- Customer
    COALESCE(dc.customer_sk, -1)    AS customer_sk,

    -- Account via person
    COALESCE(da.account_sk, -1)     AS account_sk,

    -- Date (STRING yyyy/MM/dd) based on policy_start_date (as you currently do)
    COALESCE(dd.date_sk, -1)        AS date_sk,

    -- Marketing
    COALESCE(dm.marketing_sk, -1)   AS marketing_sk,

    -- Home/Motor (via policy -> product -> home/motor)
    COALESCE(dh.home_sk, -1)        AS home_sk,
    COALESCE(dmotor.motor_sk, -1)   AS motor_sk,

    -- Measures from sat_policy
    CAST(pe.gross_revenue AS DECIMAL(18,4)) AS policy_gross_revenue_amt,
    CAST(pe.net_revenue   AS DECIMAL(18,4)) AS policy_net_revenue_amt,
    CAST(pe.renewal_amount_current_period AS DECIMAL(18,4)) AS policy_renewal_current_period_amt,
    CAST(pe.renewal_amount_next_period    AS DECIMAL(18,4)) AS policy_renewal_next_period_amt,

    -- Fact event timestamp
    pe.event_ts                     AS load_ts,
    current_timestamp()             AS created_ts,
    'ETL_SYSTEM'                    AS created_by

FROM policy_event pe

-- Hub policy
LEFT JOIN hpoc
  ON pe.policy_hash_key = hpoc.policy_hash_key

-- dim_date (full_date is STRING yyyy/MM/dd)
LEFT JOIN {catalog_name}.{gold_schema_name}.dim_date dd
  ON to_date(dd.full_date,'yyyy/MM/dd') = to_date(pe.policy_start_date)

-- dim_policy SCD2 as-of join
LEFT JOIN {catalog_name}.{gold_schema_name}.dim_policy dpo
  ON dpo.policy_id = hpoc.policy_id
 AND pe.event_ts BETWEEN dpo.effective_from_ts AND dpo.effective_to_ts

-- Policy -> Customer -> dim_customer (as-of)
LEFT JOIN rl_policy_customer rpc
  ON rpc.policy_hash_key = hpoc.policy_hash_key
LEFT JOIN {catalog_name}.{vault_schema_name}.hub_customer hc
  ON hc.customer_hash_key = rpc.customer_hash_key
LEFT JOIN {catalog_name}.{gold_schema_name}.dim_customer dc
  ON dc.customer_id = hc.customer_id
 AND pe.event_ts BETWEEN dc.effective_from_ts AND dc.effective_to_ts

-- Customer -> Person -> dim_person (as-of)
LEFT JOIN rl_customer_person rcp
  ON rcp.customer_hash_key = hc.customer_hash_key
LEFT JOIN {catalog_name}.{vault_schema_name}.hub_person hp
  ON hp.person_hash_key = rcp.person_hash_key
LEFT JOIN {catalog_name}.{gold_schema_name}.dim_person dp
  ON dp.person_id = hp.person_id
 AND pe.event_ts BETWEEN dp.effective_from_ts AND dp.effective_to_ts

-- Person -> Account -> dim_account (as-of)
LEFT JOIN rl_person_account rpa
  ON rpa.person_hash_key = hp.person_hash_key
LEFT JOIN {catalog_name}.{vault_schema_name}.hub_account ha
  ON ha.account_hash_key = rpa.account_hash_key
LEFT JOIN {catalog_name}.{gold_schema_name}.dim_account da
  ON da.account_id = ha.account_id
 AND pe.event_ts BETWEEN da.effective_from_ts AND da.effective_to_ts

-- Policy -> Product
LEFT JOIN rl_policy_product rpp
  ON rpp.policy_hash_key = hpoc.policy_hash_key

-- Product -> Home -> dim_home (as-of)
LEFT JOIN rl_product_home rph
  ON rph.product_hash_key = rpp.product_hash_key
LEFT JOIN {catalog_name}.{vault_schema_name}.hub_home hh
  ON hh.home_hash_key = rph.home_hash_key
LEFT JOIN {catalog_name}.{gold_schema_name}.dim_home dh
  ON trim(upper(dh.home_id)) = trim(upper(hh.home_id))
 AND pe.event_ts BETWEEN dh.effective_from_ts AND dh.effective_to_ts

-- Product -> Motor -> dim_motor (as-of)   ✅ fixed alias hmotor
LEFT JOIN rl_product_motor rpm
  ON rpm.product_hash_key = rpp.product_hash_key
LEFT JOIN {catalog_name}.{vault_schema_name}.hub_motor hmotor
  ON hmotor.motor_hash_key = rpm.motor_hash_key
LEFT JOIN {catalog_name}.{gold_schema_name}.dim_motor dmotor
  ON trim(upper(dmotor.motor_id)) = trim(upper(hmotor.motor_id))
 AND pe.event_ts BETWEEN dmotor.effective_from_ts AND dmotor.effective_to_ts

-- Person -> Marketing preference/engagement -> dim_marketing (as-of)
LEFT JOIN rl_person_mkt_pref rpmp
  ON rpmp.person_hash_key = hp.person_hash_key
LEFT JOIN {catalog_name}.{vault_schema_name}.hub_marketing_preference hmp
  ON hmp.marketing_preference_hash_key = rpmp.marketing_preference_hash_key

LEFT JOIN rl_person_mkt_eng rpme
  ON rpme.person_hash_key = hp.person_hash_key
LEFT JOIN {catalog_name}.{vault_schema_name}.hub_marketing_engagement hme
  ON hme.marketing_engagement_hash_key = rpme.marketing_engagement_hash_key

LEFT JOIN {catalog_name}.{gold_schema_name}.dim_marketing dm
  ON dm.marketing_preference_id = hmp.marketing_preference_id
 AND dm.marketing_engagement_id <=> hme.marketing_engagement_id
 AND pe.event_ts BETWEEN dm.effective_from_ts AND dm.effective_to_ts
""",
    "increment_keys": ['policy_sk']
}

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:720)
	at com.data

In [0]:
# FACT_LEAD_CFG_1 = {
#     "name": "fact_lead",
#     "target_table": f"{catalog_name}.{gold_schema_name}.fact_lead",
#     "stage_sql": f"""
# WITH wm AS (
#     SELECT COALESCE(MAX(watermark), TIMESTAMP('1900-01-01 00:00:00')) AS watermark
#     FROM {catalog_name}.{control_schema_name}.{watermark_table_name} WHERE table_name = '{catalog_name}.{gold_schema_name}.fact_lead'
# ),

# hl AS (
#     SELECT *
#     FROM {catalog_name}.{vault_schema_name}.hub_lead
# ),

# rl_person_lead AS (
#     SELECT *
#     FROM (
#         SELECT
#             lpl.*,
#             ROW_NUMBER() OVER (
#                 PARTITION BY lpl.lead_hash_key
#                 ORDER BY lpl.load_date DESC
#             ) AS rn_link
#         FROM {catalog_name}.{vault_schema_name}.link_person_lead lpl
#     )
#     WHERE rn_link = 1
# ),

# rl_person_mkt_pref AS (
#     SELECT *
#     FROM (
#         SELECT
#             lpmp.*,
#             ROW_NUMBER() OVER (
#                 PARTITION BY lpmp.person_hash_key
#                 ORDER BY lpmp.load_date DESC
#             ) AS rn_link
#         FROM {catalog_name}.{vault_schema_name}.link_person_marketing_preference lpmp
#     )
#     WHERE rn_link = 1
# ),

# rl_person_mkt_eng AS (
#     SELECT *
#     FROM (
#         SELECT
#             lpme.*,
#             ROW_NUMBER() OVER (
#                 PARTITION BY lpme.person_hash_key
#                 ORDER BY lpme.load_date DESC
#             ) AS rn_link
#         FROM {catalog_name}.{vault_schema_name}.link_person_marketing_engagement lpme
#     )
#     WHERE rn_link = 1
# ),

# latest_lead AS (
#     SELECT *
#     FROM (
#         SELECT
#             s.*,
#             ROW_NUMBER() OVER (
#                 PARTITION BY s.lead_hash_key
#                 ORDER BY s.load_date DESC
#             ) AS rn
#         FROM {catalog_name}.{vault_schema_name}.sat_lead s WHERE s.load_date > (SELECT watermark FROM wm)
#     )
#     WHERE rn = 1
# )

# SELECT
#     COALESCE(dp.PERSON_SK, -1) AS person_sk,
#     COALESCE(dd.date_sk, -1) AS date_sk,
#     COALESCE(dm.MARKETING_SK, -1) AS marketing_sk,
#     dp.person_id AS person_id,
#     hl.lead_id AS lead_id,
#     ll.interested_level AS interested_level,
#     CAST(ll.person_score AS FLOAT) AS person_score,
#     ll.person_status AS person_status,
#     ll.converted_date AS lead_creation_ts,
#     ll.load_date AS load_ts,
#     current_timestamp() AS created_ts,
#     'ETL_SYSTEM' AS created_by

# FROM latest_lead ll

# LEFT JOIN hl
#     ON ll.lead_hash_key = hl.lead_hash_key

# LEFT JOIN rl_person_lead rpl
#     ON rpl.lead_hash_key = hl.lead_hash_key

# LEFT JOIN {catalog_name}.{vault_schema_name}.hub_person hp
#     ON hp.person_hash_key = rpl.person_hash_key

# LEFT JOIN {catalog_name}.{gold_schema_name}.DIM_PERSON dp
#     ON dp.PERSON_ID = hp.person_id
#     AND dp.effective_to_ts = DATE('9999-12-31')

# -- added to fix and have converted date
# LEFT JOIN {catalog_name}.{gold_schema_name}.DIM_DATE dd
#     ON to_date(dd.full_date, 'yyyy/MM/dd') = to_date(ll.converted_date)

# LEFT JOIN rl_person_mkt_pref rpmp
#     ON rpmp.person_hash_key = hp.person_hash_key

# LEFT JOIN {catalog_name}.{vault_schema_name}.hub_marketing_preference mp
#     ON mp.marketing_preference_hash_key = rpmp.marketing_preference_hash_key

# LEFT JOIN rl_person_mkt_eng rpme
#     ON rpme.person_hash_key = hp.person_hash_key

# LEFT JOIN {catalog_name}.{vault_schema_name}.hub_marketing_engagement me
#     ON me.marketing_engagement_hash_key = rpme.marketing_engagement_hash_key

# LEFT JOIN {catalog_name}.{gold_schema_name}.DIM_MARKETING dm
#     ON dm.marketing_preference_id = mp.marketing_preference_id
#     AND dm.marketing_engagement_id <=> me.marketing_engagement_id
#     AND dm.effective_to_ts = TIMESTAMP('9999-12-31 00:00:00')
# """,
#     "increment_keys": ['lead_id']
# }

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:720)
	at com.data

In [0]:
FACT_QUOTE_CFG = {
    "name": "fact_quote",
    "target_table": f"{catalog_name}.{gold_schema_name}.fact_quote",
    "stage_sql": f"""
    WITH wm AS (
    SELECT COALESCE(MAX(watermark), TIMESTAMP('1900-01-01 00:00:00')) AS watermark
    FROM {catalog_name}.{control_schema_name}.{watermark_table_name} WHERE table_name = '{catalog_name}.{gold_schema_name}.fact_quote'
),

hq AS (
    SELECT *
    FROM {catalog_name}.{vault_schema_name}.hub_quote
    
),

rl_person_account AS (
    SELECT
        lpa.*,
        ROW_NUMBER() OVER (
            PARTITION BY lpa.person_hash_key
            ORDER BY lpa.load_date DESC
        ) AS rn_link
    FROM {catalog_name}.{vault_schema_name}.link_person_account lpa
),

-- Ranked links (avoid fan-out)
rl_quote_person AS (
    SELECT
        lqp.*,
        ROW_NUMBER() OVER (
            PARTITION BY lqp.quote_hash_key
            ORDER BY lqp.load_date DESC
        ) AS rn_link
    FROM {catalog_name}.{vault_schema_name}.link_quote_person lqp
),

rl_quote_product AS (
    SELECT
        lpr.*,
        ROW_NUMBER() OVER (
            PARTITION BY lpr.quote_hash_key
            ORDER BY lpr.load_date DESC
        ) AS rn_link
    FROM {catalog_name}.{vault_schema_name}.link_quote_product lpr
),

rl_product_home AS (
    SELECT
        lph.*,
        ROW_NUMBER() OVER (
            PARTITION BY lph.product_hash_key
            ORDER BY lph.load_date DESC
        ) AS rn_link
    FROM {catalog_name}.{vault_schema_name}.link_product_home lph
),

rl_product_motor AS (
    SELECT
        lpm.*,
        ROW_NUMBER() OVER (
            PARTITION BY lpm.product_hash_key
            ORDER BY lpm.load_date DESC
        ) AS rn_link
    FROM {catalog_name}.{vault_schema_name}.link_product_motor lpm
),

rl_customer_person AS (
    SELECT
        lcp.*,
        ROW_NUMBER() OVER (
            PARTITION BY lcp.person_hash_key
            ORDER BY lcp.load_date DESC
        ) AS rn_link
    FROM {catalog_name}.{vault_schema_name}.link_customer_person lcp
),

rl_person_mkt_pref AS (
    SELECT
        lpmp.*,
        ROW_NUMBER() OVER (
            PARTITION BY lpmp.person_hash_key
            ORDER BY lpmp.load_date DESC
        ) AS rn_link
    FROM {catalog_name}.{vault_schema_name}.link_person_marketing_preference lpmp
),

rl_person_mkt_eng AS (
    SELECT
        lpme.*,
        ROW_NUMBER() OVER (
            PARTITION BY lpme.person_hash_key
            ORDER BY lpme.load_date DESC
        ) AS rn_link
    FROM {catalog_name}.{vault_schema_name}.link_person_marketing_engagement lpme
),

-- Latest satellites
latest_quote AS (
    SELECT
        s.*,
        ROW_NUMBER() OVER (
            PARTITION BY s.quote_hash_key
            ORDER BY s.load_date DESC
        ) AS rn
    FROM {catalog_name}.{vault_schema_name}.sat_quote s WHERE s.load_date > (SELECT watermark FROM wm)
)

SELECT
    -- Person (Quote → Person link)
    COALESCE(dp.PERSON_SK, -1) AS person_sk,

    -- Customer via Person→Customer link
    COALESCE(dc.CUSTOMER_SK, -1) AS customer_sk,

    -- Account via Person→Account (optional, if available)
    COALESCE(da.ACCOUNT_SK, -1) AS account_sk,

    -- Marketing
    COALESCE(dm.MARKETING_SK, -1) AS marketing_sk,

    -- Date
    COALESCE(dd.date_sk, -1) AS date_sk,

    -- Home
    COALESCE(dh.HOME_SK, -1) AS home_sk,

    -- Motor
    COALESCE(dmotor.motor_sk, -1) AS motor_sk,

    -- Quote attributes / measures
    dp.person_id AS person_id,
    hq.quote_id AS quote_id,
    lq.quote_number AS quote_number,
    lq.quote_status AS quote_status,
    CAST(lq.gross_revenue AS DECIMAL(18,4)) AS quote_gross_revenue_amt,
    CAST(lq.net_revenue AS DECIMAL(18,4)) AS quote_net_revenue_amt,
    CAST(lq.renewal_amt_current_period AS DECIMAL(18,4)) AS quote_renewal_current_period_amt,
    CAST(lq.renewal_amt_next_period AS DECIMAL(18,4)) AS quote_renewal_next_period_amt,
    lq.load_date AS load_ts,
    current_timestamp() AS created_ts,
    'ETL_SYSTEM' AS created_by

FROM latest_quote lq

LEFT JOIN hq
    ON lq.quote_hash_key = hq.quote_hash_key AND lq.rn = 1

-- Person
LEFT JOIN rl_quote_person rqp
    ON rqp.quote_hash_key = hq.quote_hash_key AND rqp.rn_link = 1
LEFT JOIN {catalog_name}.{vault_schema_name}.hub_person hp
    ON hp.person_hash_key = rqp.person_hash_key
LEFT JOIN {catalog_name}.{gold_schema_name}.DIM_PERSON dp
    ON dp.PERSON_ID = hp.person_id
    AND dp.effective_to_ts = TIMESTAMP('9999-12-31 00:00:00')
LEFT JOIN {catalog_name}.{gold_schema_name}.DIM_DATE dd
    ON to_date(dd.full_date, 'yyyy/MM/dd') = to_date(lq.load_date)

-- Customer via Person→Customer
LEFT JOIN rl_customer_person rcp
    ON rcp.person_hash_key = hp.person_hash_key AND rcp.rn_link = 1
LEFT JOIN {catalog_name}.{vault_schema_name}.hub_customer hc
    ON hc.customer_hash_key = rcp.customer_hash_key
LEFT JOIN {catalog_name}.{gold_schema_name}.DIM_CUSTOMER dc
    ON dc.CUSTOMER_ID = hc.customer_id
    AND dc.effective_to_ts = TIMESTAMP('9999-12-31 00:00:00')

-- Account via Person→Account
LEFT JOIN rl_person_account rpa
    ON rpa.person_hash_key = hp.person_hash_key
    AND rpa.rn_link = 1
LEFT JOIN {catalog_name}.{vault_schema_name}.hub_account ha
    ON ha.account_hash_key = rpa.account_hash_key
LEFT JOIN {catalog_name}.{gold_schema_name}.DIM_ACCOUNT da
    ON da.ACCOUNT_ID = ha.account_id
    AND da.effective_to_ts = TIMESTAMP('9999-12-31 00:00:00')

-- Policy → Product
LEFT JOIN rl_quote_product rpp
    ON rpp.quote_hash_key = hq.quote_hash_key
    AND rpp.rn_link = 1

--LEFT JOIN {catalog_name}.{vault_schema_name}.hub_product hpr ON hpr.product_hash_key = rpp.product_hash_key

-- Product → Home → DIM_HOME
LEFT JOIN rl_product_home rph
    ON rph.product_hash_key = rpp.product_hash_key AND rph.rn_link = 1
LEFT JOIN {catalog_name}.{vault_schema_name}.hub_home hh
    ON hh.home_hash_key = rph.home_hash_key
LEFT JOIN {catalog_name}.{gold_schema_name}.DIM_HOME dh
    ON TRIM(UPPER(dh.HOME_ID)) = TRIM(UPPER(hh.home_id))
    AND dh.effective_to_ts = TIMESTAMP('9999-12-31 00:00:00')

-- Product → Motor → DIM_MOTOR
LEFT JOIN rl_product_motor rpm
    ON rpm.product_hash_key = rpp.product_hash_key AND rpm.rn_link = 1
LEFT JOIN {catalog_name}.{vault_schema_name}.hub_motor hmotor
    ON hmotor.motor_hash_key = rpm.motor_hash_key
LEFT JOIN {catalog_name}.{gold_schema_name}.DIM_MOTOR dmotor
    ON TRIM(UPPER(dmotor.MOTOR_ID)) = TRIM(UPPER(hmotor.motor_id))
    AND dmotor.effective_to_ts = TIMESTAMP('9999-12-31 00:00:00')

-- Marketing
LEFT JOIN rl_person_mkt_pref rpmp
    ON rpmp.person_hash_key = hp.person_hash_key AND rpmp.rn_link = 1
LEFT JOIN {catalog_name}.{vault_schema_name}.hub_marketing_preference hmp
    ON hmp.marketing_preference_hash_key = rpmp.marketing_preference_hash_key
LEFT JOIN rl_person_mkt_eng rpme
    ON rpme.person_hash_key = hp.person_hash_key AND rpme.rn_link = 1
LEFT JOIN {catalog_name}.{vault_schema_name}.hub_marketing_engagement hme
    ON hme.marketing_engagement_hash_key = rpme.marketing_engagement_hash_key
LEFT JOIN {catalog_name}.{gold_schema_name}.DIM_MARKETING dm
    ON dm.marketing_preference_id = hmp.marketing_preference_id
    AND dm.marketing_engagement_id <=> hme.marketing_engagement_id
    AND dm.effective_to_ts = TIMESTAMP('9999-12-31 00:00:00')
""",
    "increment_keys": ['quote_id']
}

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:720)
	at com.data

In [0]:
# FACT_POLICY_CFG_1 = {
#     "name": "fact_policy",
#     "target_table": f"{catalog_name}.{gold_schema_name}.fact_policy",
#     "stage_sql": f"""
# WITH wm AS (
#     SELECT COALESCE(MAX(watermark), TIMESTAMP('1900-01-01 00:00:00')) AS watermark
#     FROM {catalog_name}.{control_schema_name}.{watermark_table_name} WHERE table_name = '{catalog_name}.{gold_schema_name}.fact_policy'
# ),

# hpoc AS (
#     SELECT *
#     FROM {catalog_name}.{vault_schema_name}.hub_policy   
# ),

# rl_person_account AS (
#     SELECT
#         lpa.*,
#         ROW_NUMBER() OVER (
#             PARTITION BY lpa.person_hash_key
#             ORDER BY lpa.load_date DESC
#         ) AS rn_link
#     FROM {catalog_name}.{vault_schema_name}.link_person_account lpa
# ),

# rl_policy_customer AS (
#     SELECT
#         lpc.*,
#         ROW_NUMBER() OVER (
#             PARTITION BY lpc.policy_hash_key
#             ORDER BY lpc.load_date DESC
#         ) AS rn_link
#     FROM {catalog_name}.{vault_schema_name}.link_policy_customer lpc
# ),

# rl_policy_product AS (
#     SELECT
#         lpp.*,
#         ROW_NUMBER() OVER (
#             PARTITION BY lpp.policy_hash_key
#             ORDER BY lpp.load_date DESC
#         ) AS rn_link
#     FROM {catalog_name}.{vault_schema_name}.link_policy_product lpp
# ),

# rl_product_home AS (
#     SELECT
#         lph.*,
#         ROW_NUMBER() OVER (
#             PARTITION BY lph.product_hash_key
#             ORDER BY lph.load_date DESC
#         ) AS rn_link
#     FROM {catalog_name}.{vault_schema_name}.link_product_home lph
# ),

# rl_product_motor AS (
#     SELECT
#         lpm.*,
#         ROW_NUMBER() OVER (
#             PARTITION BY lpm.product_hash_key
#             ORDER BY lpm.load_date DESC
#         ) AS rn_link
#     FROM {catalog_name}.{vault_schema_name}.link_product_motor lpm
# ),

# rl_customer_person AS (
#     SELECT
#         lcp.*,
#         ROW_NUMBER() OVER (
#             PARTITION BY lcp.customer_hash_key
#             ORDER BY lcp.load_date DESC
#         ) AS rn_link
#     FROM {catalog_name}.{vault_schema_name}.link_customer_person lcp
# ),

# latest_policy AS (
#     SELECT
#         s.*,
#         ROW_NUMBER() OVER (
#             PARTITION BY s.policy_hash_key
#             ORDER BY s.load_date DESC
#         ) AS rn
#     FROM {catalog_name}.{vault_schema_name}.sat_policy s WHERE s.load_date > (SELECT watermark FROM wm)
# ),

# rl_person_mkt_pref AS (
#     SELECT
#         lpmp.*,
#         ROW_NUMBER() OVER (
#             PARTITION BY lpmp.person_hash_key
#             ORDER BY lpmp.load_date DESC
#         ) AS rn_link
#     FROM {catalog_name}.{vault_schema_name}.link_person_marketing_preference lpmp
# ),

# rl_person_mkt_eng AS (
#     SELECT
#         lpme.*,
#         ROW_NUMBER() OVER (
#             PARTITION BY lpme.person_hash_key
#             ORDER BY lpme.load_date DESC
#         ) AS rn_link
#     FROM {catalog_name}.{vault_schema_name}.link_person_marketing_engagement lpme
# )

# SELECT
#     -- Policy
#     COALESCE(dpo.POLICY_SK, -1) AS policy_sk,

#     -- Person via Customer→Person
#     COALESCE(dp.PERSON_SK, -1) AS person_sk,
#     hp.person_id AS person_id,

#     -- Customer (Policy→Customer)
#     COALESCE(dc.CUSTOMER_SK, -1) AS customer_sk,
#     --hc.customer_id AS CUSTOMER_ID,

#     -- Account via Person→Account
#     COALESCE(da.ACCOUNT_SK, -1) AS account_sk,
#     --ha.account_id AS ACCOUNT_ID,

#     -- Date
#     COALESCE(dd.date_sk, -1) AS date_sk,

#     -- Marketing
#     COALESCE(dm.MARKETING_SK, -1) AS marketing_sk,

#     -- Home via Policy→Product→Home (current DIM row)
#     COALESCE(dh.HOME_SK, -1) AS home_sk,
#     --hh.home_id AS HOME_ID,

#     -- Motor via Policy→Product→Motor (current DIM row)
#     COALESCE(dmotor.MOTOR_SK, -1) AS motor_sk,

#     --hpoc.policy_id AS policy_id,

#     --hmotor.motor_id AS MOTOR_ID,

#     --hmp.marketing_preference_id,
#     --hme.marketing_engagement_id,

#     -- Measures from sat_policy
#     CAST(lp.gross_revenue AS DECIMAL(18,4)) AS policy_gross_revenue_amt,
#     CAST(lp.net_revenue   AS DECIMAL(18,4)) AS policy_net_revenue_amt,
#     CAST(lp.renewal_amount_current_period AS DECIMAL(18,4)) AS policy_renewal_current_period_amt,
#     CAST(lp.renewal_amount_next_period    AS DECIMAL(18,4)) AS policy_renewal_next_period_amt,
#     lp.load_date as load_ts,
#     current_timestamp() AS created_ts,
#     'ETL_SYSTEM' AS created_by

# FROM latest_policy lp

# -- Policy satellite and Date
# LEFT JOIN hpoc
#     ON lp.policy_hash_key = hpoc.policy_hash_key AND lp.rn = 1
# LEFT JOIN {catalog_name}.{gold_schema_name}.DIM_DATE dd
#     ON to_date(dd.full_date,'yyyy/MM/dd') = to_date(lp.policy_start_date)

# -- DIM_POLICY lookup by policy_id
# LEFT JOIN {catalog_name}.{gold_schema_name}.DIM_POLICY dpo
#     ON dpo.POLICY_ID = hpoc.policy_id AND dpo.effective_to_ts = TIMESTAMP('9999-12-31 00:00:00')

# -- Policy→Customer
# LEFT JOIN rl_policy_customer rpc
#     ON rpc.policy_hash_key = hpoc.policy_hash_key AND rpc.rn_link = 1
# LEFT JOIN {catalog_name}.{vault_schema_name}.hub_customer hc
#     ON hc.customer_hash_key = rpc.customer_hash_key
# LEFT JOIN {catalog_name}.{gold_schema_name}.DIM_CUSTOMER dc
#     ON dc.CUSTOMER_ID = hc.customer_id AND dc.effective_to_ts = TIMESTAMP('9999-12-31 00:00:00')

# -- Customer → Person
# LEFT JOIN rl_customer_person rcp
#     ON rcp.customer_hash_key = hc.customer_hash_key AND rcp.rn_link = 1
# LEFT JOIN {catalog_name}.{vault_schema_name}.hub_person hp
#     ON hp.person_hash_key = rcp.person_hash_key
# LEFT JOIN {catalog_name}.{gold_schema_name}.DIM_PERSON dp
#     ON dp.PERSON_ID = hp.person_id AND dp.effective_to_ts = TIMESTAMP('9999-12-31 00:00:00')

# -- Person → Account (optional)
# LEFT JOIN rl_person_account rpa
#     ON rpa.person_hash_key = hp.person_hash_key
#     AND rpa.rn_link = 1
# LEFT JOIN {catalog_name}.{vault_schema_name}.hub_account ha
#     ON ha.account_hash_key = rpa.account_hash_key
# LEFT JOIN {catalog_name}.{gold_schema_name}.DIM_ACCOUNT da
#     ON da.ACCOUNT_ID = ha.account_id AND da.effective_to_ts = TIMESTAMP('9999-12-31 00:00:00')

# -- Policy → Product
# LEFT JOIN rl_policy_product rpp
#     ON rpp.policy_hash_key = hpoc.policy_hash_key AND rpp.rn_link = 1
# --LEFT JOIN {catalog_name}.{vault_schema_name}.hub_product hpr ON hpr.product_hash_key = rpp.product_hash_key

# -- Product → Home (ranked) → DIM_HOME
# LEFT JOIN rl_product_home rph
#     ON rph.product_hash_key = rpp.product_hash_key AND rph.rn_link = 1
# LEFT JOIN {catalog_name}.{vault_schema_name}.hub_home hh
#     ON hh.home_hash_key = rph.home_hash_key
# LEFT JOIN {catalog_name}.{gold_schema_name}.DIM_HOME dh
#     ON TRIM(UPPER(dh.HOME_ID)) = TRIM(UPPER(hh.home_id)) AND dh.effective_to_ts = TIMESTAMP('9999-12-31 00:00:00')

# -- Product → Motor (ranked) → DIM_MOTOR
# LEFT JOIN rl_product_motor rpm
#     ON rpm.product_hash_key = rpp.product_hash_key AND rpm.rn_link = 1
# LEFT JOIN {catalog_name}.{vault_schema_name}.hub_motor hmotor
#     ON hmotor.motor_hash_key = rpm.motor_hash_key
# LEFT JOIN {catalog_name}.{gold_schema_name}.DIM_MOTOR dmotor
#     ON TRIM(UPPER(dmotor.MOTOR_ID)) = TRIM(UPPER(hmotor.motor_id)) AND dmotor.effective_to_ts = TIMESTAMP('9999-12-31 00:00:00')

# -- Person-time Marketing
# LEFT JOIN rl_person_mkt_pref rpmp
#     ON rpmp.person_hash_key = hp.person_hash_key AND rpmp.rn_link = 1
# LEFT JOIN {catalog_name}.{vault_schema_name}.hub_marketing_preference hmp
#     ON hmp.marketing_preference_hash_key = rpmp.marketing_preference_hash_key
# LEFT JOIN rl_person_mkt_eng rpme
#     ON rpme.person_hash_key = hp.person_hash_key AND rpme.rn_link = 1
# LEFT JOIN {catalog_name}.{vault_schema_name}.hub_marketing_engagement hme
#     ON hme.marketing_engagement_hash_key = rpme.marketing_engagement_hash_key
# LEFT JOIN {catalog_name}.{gold_schema_name}.DIM_MARKETING dm
#     ON dm.marketing_preference_id = hmp.marketing_preference_id
#     AND dm.marketing_engagement_id <=> hme.marketing_engagement_id
#     AND dm.effective_to_ts = TIMESTAMP('9999-12-31 00:00:00')
# """,
#     "increment_keys": ['policy_sk']
# }

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:720)
	at com.data